# Exploratory Data Analysis (EDA): `transaction_data.csv`

This notebook explores transaction behavior with a focus on:

- data quality and missing values
- transaction value and quantity distributions
- country-level patterns
- time-based trends (hour/day/month)
- top products and users
- return/cancellation patterns

> Note: the dataset is very large, so a row cap is configurable for faster iteration.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [ ]:
# Config
CSV_PATH = "transaction_data.csv"
NROWS = 300_000  # set to None to read full file
RANDOM_SEED = 42

raw_df = pd.read_csv(CSV_PATH, nrows=NROWS)
print(f"Loaded shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Basic schema and null profile
raw_df.info()

nulls = raw_df.isna().sum().sort_values(ascending=False)
null_pct = (nulls / len(raw_df) * 100).round(2)
null_report = pd.DataFrame({"null_count": nulls, "null_pct": null_pct})
null_report

In [ ]:
# Cleaning + feature engineering

df = raw_df.copy()

# Coerce numerics
df["UserId"] = pd.to_numeric(df["UserId"], errors="coerce")
df["TransactionId"] = pd.to_numeric(df["TransactionId"], errors="coerce")
df["ItemCode"] = pd.to_numeric(df["ItemCode"], errors="coerce")
df["NumberOfItemsPurchased"] = pd.to_numeric(df["NumberOfItemsPurchased"], errors="coerce")
df["CostPerItem"] = pd.to_numeric(df["CostPerItem"], errors="coerce")

# Parse timestamp format like "Sat Feb 02 12:50:00 IST 2019"
df["TransactionTimeParsed"] = pd.to_datetime(df["TransactionTime"], errors="coerce", utc=False)

# Derived fields
df["TotalValue"] = df["NumberOfItemsPurchased"] * df["CostPerItem"]
df["AbsTotalValue"] = df["TotalValue"].abs()
df["IsGuestOrUnknownUser"] = (df["UserId"] == -1).astype(int)
df["IsReturnOrCancellation"] = (df["NumberOfItemsPurchased"] < 0).astype(int)

# Time fields
parsed = df["TransactionTimeParsed"]
df["Hour"] = parsed.dt.hour
df["DayOfWeek"] = parsed.dt.day_name()
df["Month"] = parsed.dt.to_period("M").astype(str)

print(f"Cleaned shape: {df.shape}")
df.head()

In [ ]:
# Quick summary stats
summary_cols = [
    "NumberOfItemsPurchased",
    "CostPerItem",
    "TotalValue",
    "AbsTotalValue",
    "IsGuestOrUnknownUser",
    "IsReturnOrCancellation",
]
df[summary_cols].describe(include="all").T

## 1) Distributions: Quantity, Price, and Transaction Value

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(df["NumberOfItemsPurchased"].dropna(), bins=80, color="#4C78A8")
axes[0].set_title("NumberOfItemsPurchased")
axes[0].set_xlabel("items")

axes[1].hist(df["CostPerItem"].dropna(), bins=80, color="#F58518")
axes[1].set_title("CostPerItem")
axes[1].set_xlabel("cost")

# Clip long tail for visibility
val_clip = df["AbsTotalValue"].dropna().clip(upper=df["AbsTotalValue"].quantile(0.99))
axes[2].hist(val_clip, bins=80, color="#54A24B")
axes[2].set_title("AbsTotalValue (clipped @99th pct)")
axes[2].set_xlabel("abs total value")

plt.tight_layout()
plt.show()

In [ ]:
# Return/cancellation profile
return_rate = df["IsReturnOrCancellation"].mean() * 100
guest_rate = df["IsGuestOrUnknownUser"].mean() * 100
print(f"Return/Cancellation rate: {return_rate:.2f}%")
print(f"Guest/Unknown user rate (UserId=-1): {guest_rate:.2f}%")

ret_by_country = (
    df.groupby("Country", dropna=False)["IsReturnOrCancellation"]
      .mean()
      .sort_values(ascending=False)
      .head(15)
)

plt.figure(figsize=(10, 5))
ret_by_country.sort_values().plot(kind="barh", color="#E45756")
plt.title("Top 15 Countries by Return/Cancellation Rate")
plt.xlabel("rate")
plt.ylabel("country")
plt.tight_layout()
plt.show()

## 2) Geography: Country-level transaction patterns

In [ ]:
country_agg = (
    df.groupby("Country", dropna=False)
      .agg(
          transactions=("TransactionId", "count"),
          total_items=("NumberOfItemsPurchased", "sum"),
          total_value=("TotalValue", "sum"),
          avg_value=("TotalValue", "mean"),
      )
      .sort_values("transactions", ascending=False)
)
country_agg.head(10)

In [ ]:
top_countries = country_agg.head(12).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_countries["transactions"].sort_values().plot(kind="barh", ax=axes[0], color="#72B7B2")
axes[0].set_title("Top Countries by Transaction Count")
axes[0].set_xlabel("transactions")

(top_countries["total_value"]
 .sort_values()
 .plot(kind="barh", ax=axes[1], color="#B279A2"))
axes[1].set_title("Top Countries by Total Value")
axes[1].set_xlabel("total value")

plt.tight_layout()
plt.show()

## 3) Time analysis: Hourly, weekday, and monthly trends

In [ ]:
hourly = df.groupby("Hour")["TransactionId"].count()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = df.groupby("DayOfWeek")["TransactionId"].count().reindex(weekday_order)
monthly = df.groupby("Month")["TransactionId"].count().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

hourly.plot(ax=axes[0], marker="o", color="#4C78A8")
axes[0].set_title("Transactions by Hour")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("count")

weekday.plot(kind="bar", ax=axes[1], color="#F58518")
axes[1].set_title("Transactions by Day of Week")
axes[1].set_xlabel("")

monthly.plot(ax=axes[2], marker="o", color="#54A24B")
axes[2].set_title("Transactions by Month")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

## 4) Product and user behavior

In [ ]:
top_items = (
    df.groupby("ItemDescription", dropna=False)
      .agg(transactions=("TransactionId", "count"), total_items=("NumberOfItemsPurchased", "sum"))
      .sort_values("transactions", ascending=False)
      .head(15)
)

top_users = (
    df[df["UserId"] != -1]
      .groupby("UserId")
      .agg(transactions=("TransactionId", "count"), spend=("TotalValue", "sum"))
      .sort_values("transactions", ascending=False)
      .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_items["transactions"].sort_values().plot(kind="barh", ax=axes[0], color="#B279A2")
axes[0].set_title("Top 15 ItemDescriptions by Transaction Count")
axes[0].set_xlabel("transactions")

(top_users["transactions"].sort_values().plot(kind="barh", ax=axes[1], color="#FF9DA6"))
axes[1].set_title("Top 15 Users by Transaction Count (excluding UserId=-1)")
axes[1].set_xlabel("transactions")

plt.tight_layout()
plt.show()

top_items.head(), top_users.head()

## 5) Outliers and correlation

In [ ]:
numeric_cols = ["NumberOfItemsPurchased", "CostPerItem", "TotalValue", "AbsTotalValue", "Hour", "IsReturnOrCancellation", "IsGuestOrUnknownUser"]

sample_df = df[numeric_cols].dropna().sample(min(20000, len(df)), random_state=RANDOM_SEED)

# Correlation matrix heatmap (matplotlib only)
corr = sample_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Heatmap (sampled)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Scatter for value vs quantity (sampled)
scatter_df = df[["NumberOfItemsPurchased", "TotalValue", "IsReturnOrCancellation"]].dropna().sample(min(15000, len(df)), random_state=RANDOM_SEED)
colors = np.where(scatter_df["IsReturnOrCancellation"] == 1, "#E45756", "#4C78A8")

plt.figure(figsize=(8, 5))
plt.scatter(scatter_df["NumberOfItemsPurchased"], scatter_df["TotalValue"], s=8, alpha=0.25, c=colors)
plt.title("TotalValue vs NumberOfItemsPurchased (sampled)")
plt.xlabel("NumberOfItemsPurchased")
plt.ylabel("TotalValue")
plt.tight_layout()
plt.show()

## 6) Key takeaways (auto-generated checks)

Run the cell below after executing all plots to print compact EDA observations.

In [ ]:
top_country = country_agg.index[0] if len(country_agg) else "NA"
ret_rate = df["IsReturnOrCancellation"].mean() * 100
guest_rate = df["IsGuestOrUnknownUser"].mean() * 100

print("EDA quick observations")
print("-" * 60)
print(f"Rows analyzed: {len(df):,}")
print(f"Top country by transactions: {top_country}")
print(f"Return/Cancellation rate: {ret_rate:.2f}%")
print(f"Guest/Unknown user rate: {guest_rate:.2f}%")
print(f"Median quantity: {df['NumberOfItemsPurchased'].median():.2f}")
print(f"Median cost/item: {df['CostPerItem'].median():.2f}")
print(f"Median abs transaction value: {df['AbsTotalValue'].median():.2f}")